In [1]:
#Import your needed modules
import pandas as pd
import numpy as np
#psycopg2 will be imported later when needed

In [2]:
#Collect a range of years (this way is easier to alter if want to update)
years = range(2016, 2026)

#Collect the regular season data in one giant dataframe looping through the years above (Index true turns continues listing)
reg_data = pd.concat([
    pd.read_csv(f"../rawdata/reg_season/stats_player_reg_{year}.csv"
                )
    for year in years       #Inner loop
], ignore_index=True)

#Collect the post season data in one giant dataframe looping through the years above (Index true turns continues listing)
post_data =  pd.concat([
    pd.read_csv(f"../rawdata/post_season/stats_player_post_{year}.csv"
                )
    for year in years       #Inner loop
], ignore_index=True)

In [3]:
#Concat the two dataframes into one for simplicity
qb_data = pd.concat([reg_data, post_data], ignore_index=True)

In [4]:
#Print first 5 rows to confirm it is working (change to tail to confrim latter rows are added)
qb_data.head()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,season_type,recent_team,games,...,pt_out_of_bounds,pt_downed,pt_touchback,pt_fair_caught,pt_returned,pt_return_yards,pt_return_tds,pt_net_yards,fantasy_points,fantasy_points_ppr
0,00-0004091,P.Dawson,Phil Dawson,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2016,REG,SF,16,...,0,0,0,0,0,0,0,0,0.00,0.00
1,00-0016919,A.Vinatieri,Adam Vinatieri,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2016,REG,IND,16,...,0,0,0,0,0,0,0,0,0.00,0.00
2,00-0019596,T.Brady,Tom Brady,QB,QB,https://static.www.nfl.com/image/private/f_aut...,2016,REG,NE,12,...,0,0,0,0,0,0,0,0,258.56,258.56
3,00-0019646,S.Janikowski,Sebastian Janikowski,K,SPEC,https://static.www.nfl.com/image/private/f_aut...,2016,REG,LV,16,...,0,0,0,0,0,0,0,0,0.00,0.00
4,00-0019714,S.Lechler,Shane Lechler,P,SPEC,https://static.www.nfl.com/image/private/f_aut...,2016,REG,HOU,16,...,5,3,3,13,48,477,1,2886,0.00,0.00


In [5]:
#Sort the data to only acquire columns applying to the QB position
qb_data_needed = ["player_id", "player_name", "player_display_name", "position", "position_group",
                  "headshot_url", "season", "season_type", "recent_team", "games",
                  "completions", "attempts", "passing_yards", "passing_tds", "passing_interceptions",
                  "sacks_suffered", "sack_yards_lost", "sack_fumbles", "sack_fumbles_lost", "passing_air_yards", 
                  "passing_yards_after_catch", "passing_first_downs", "passing_epa", "passing_cpoe", "passing_2pt_conversions",
                  "pacr", "passing_10", "passing_16", "passing_20", "passing_40", "carries", "rushing_yards", "rushing_tds",
                  "rushing_fumbles", "rushing_fumbles_lost", "rushing_first_downs", "rushing_epa", "rushing_2pt_conversions",
                  "rushing_10", "rushing_12", "rushing_20", "rushing_40", "receiving_yards", "receiving_tds", "fumbles_total", 
                  "fumbles_lost_total", "fantasy_points", "fantasy_points_ppr"]

In [6]:
#Filter the data frame with the columns we chose above
qb_data = qb_data[qb_data_needed]

In [7]:
#Futher filter the data to only include the QB position
qb_data = qb_data[(qb_data["position"] == "QB")]

In [8]:
#Check results this time using tail to get the last 5 rows
qb_data.tail()

,player_id,player_name,player_display_name,position,position_group,headshot_url,season,season_type,recent_team,games,...,rushing_10,rushing_12,rushing_20,rushing_40,receiving_yards,receiving_tds,fumbles_total,fumbles_lost_total,fantasy_points,fantasy_points_ppr
24170,00-0039150,B.Young,Bryce Young,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,POST,CAR,1,...,1,1,0,0,0,0,0,0,20.96,20.96
24172,00-0039163,C.Stroud,C.J. Stroud,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,POST,HOU,2,...,1,0,0,0,0,0,5,2,13.58,13.58
24201,00-0039732,B.Nix,Bo Nix,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,POST,DEN,1,...,1,0,0,0,0,0,1,0,24.06,24.06
24228,00-0039851,D.Maye,Drake Maye,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,POST,NE,4,...,6,5,2,0,0,0,7,4,64.92,64.92
24239,00-0039918,C.Williams,Caleb Williams,QB,QB,https://static.www.nfl.com/image/upload/f_auto...,2025,POST,CHI,2,...,2,1,1,0,0,0,1,0,38.72,38.72


In [9]:
#Leverage new qb statistics with existing features in the dataset
qb_data["completion_pct"] = round((qb_data["completions"] / qb_data["attempts"]) * 100, 2)
qb_data["td_to_int_ratio"] = round(qb_data["passing_tds"] / qb_data["passing_interceptions"], 2)
qb_data["total_turnovers"] =  qb_data["passing_interceptions"] + qb_data["fumbles_lost_total"]
qb_data["yards_per_att"] = round(qb_data["passing_yards"] / qb_data["attempts"], 2)
qb_data["td_per_att"] = round(qb_data["passing_tds"] / qb_data["attempts"], 2)
qb_data["air_yds_att"] = round(qb_data["passing_air_yards"] / qb_data["attempts"], 2)
qb_data["yds_per_rush"] = round(qb_data["rushing_yards"] / qb_data["carries"], 2)
qb_data["total_yards"] = qb_data["passing_yards"] + qb_data["rushing_yards"] + qb_data["receiving_yards"]
qb_data["total_tds"] = qb_data["passing_tds"] + qb_data["rushing_tds"] + qb_data["receiving_tds"]
qb_data["total_epa"] = round(qb_data["passing_epa"] + qb_data["rushing_epa"],2)
qb_data["total_dropbacks"] = qb_data["sacks_suffered"] + qb_data["attempts"]
qb_data["epa_per_dropback"] = round(qb_data["total_epa"] / qb_data["total_dropbacks"], 2)

In [10]:
#Calculate the Passer Rating for each QB based on the NFL formulas

#Passer Rating Formula
a = ((qb_data["completions"] / qb_data["attempts"]) - 0.3) * 5
b = ((qb_data["passing_yards"] / qb_data["attempts"]) - 3) * 0.25
c = (qb_data["passing_tds"] / qb_data["attempts"]) * 20
d = 2.375 - ((qb_data["passing_interceptions"] / qb_data["attempts"]) * 25)

# Apply NFL bounds (0 to 2.375)
a = a.clip(0, 2.375)
b = b.clip(0, 2.375)
c = c.clip(0, 2.375)
d = d.clip(0, 2.375)

# Final Passer Rating
qb_data["passer_rating"] = round(((a + b + c + d) / 6) * 100,2)

In [11]:
#Take care of Na and division by zero here
qb_data = qb_data.fillna(0)
qb_data = qb_data.replace([np.inf, -np.inf], 0)

In [12]:
#Check to make sure the error handling worked
print(qb_data.isnull().sum().sum())

0


In [13]:
#Double check the datatypes before connecting it to your server
qb_data.dtypes

player_id                  str
player_name                str
player_display_name        str
position                   str
position_group             str
                        ...   
total_tds                int64
total_epa              float64
total_dropbacks          int64
epa_per_dropback       float64
passer_rating          float64
Length: 61, dtype: object

In [14]:
#Export the cleaned up dataset into a regular csv file to import into sql database created.
qb_data.to_csv("../cleandata/cleaned_qb_stats.csv", index=False)

In [15]:
#Connects to my postgresql server to create an excel file
#Might need to adjust to your configuration
import psycopg2

conn = psycopg2.connect(
    dbname = "postgres",
    user = "postgres",
    password = "123",
    host = "localhost",
    port = "5432",
)

In [16]:
#List of views created in my qb_data_views.sql file
views = [
    "top_25_reg_pass_yards",
    "avg_yards_reg_season",
    "top_25_reg_passer_rating",
    "avg_yds_thrown_downfield_reg",
    "top_25_reg_total_yards",
    "avg_qb_rush_reg_yards_season",
    "best_td_int_per_reg_season",
    "top_playoff_performers",
    "combined_reg_post_stats",
    "reg_vs_post_game_avg",
    "epa_vs_tds_scored"
]

In [17]:
#Create the excel xlsx file used for pivot tables (Can adjust to just a csv file if wanted, would need to adjust engine below)
analysis_ready_data = "../analysis_ready_data/qb_data_views.xlsx"

In [ ]:
#Write to the excel xlsx file to add the original data and the views
with pd.ExcelWriter(analysis_ready_data, engine="openpyxl") as writer:
    #Original data set added as the first sheet 
    qb_data.to_excel(writer, sheet_name="qb_data_clean", index=False)

    #Add the created views as there own seperate sheet
    for view in views:
        df = pd.read_sql(f"SELECT * FROM {view};", conn)
        df.to_excel(writer, sheet_name=view, index=False)

#Close the connection
conn.close()

C:\Users\Mike\AppData\Local\Temp\ipykernel_21064\3813875182.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {view};", conn)
